# Classical ML Models — Rain in Australia

This notebook trains and evaluates four classical ML models for predicting **RainTomorrow** (binary):
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

All model pipelines are defined in `classical_models.py`.

**Primary evaluation metric:** ROC-AUC (per cahier de charge).

## 1 — Imports & Constants

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import RocCurveDisplay
from classical_models import get_all_models, evaluate_model, FEATURE_COLUMNS, TARGET_RAIN

DATA_PATH   = "data/clean_data.csv"
MODELS_DIR  = "saved_models/"
os.makedirs(MODELS_DIR, exist_ok=True)

print("Imports OK")

## 2 — Data Loading

In [ ]:
# Load a small HEAD for schema validation only
df_schema = pd.read_csv(DATA_PATH, nrows=5)
print("Columns:", df_schema.columns.tolist())
print("Shape preview:", df_schema.shape)

# Full load (uncomment to train)
# df = pd.read_csv(DATA_PATH)
# X = df[FEATURE_COLUMNS]
# y = df[TARGET_RAIN]
# print(f"Full dataset: {X.shape}")
# print(f"Rain prevalence: {y.mean():.2%}")

## 3 — Train / Test Split

In [ ]:
# Stratified split to preserve class balance
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.2,
#     random_state=42,
#     stratify=y      # Critical: preserve ~22% rain ratio
# )
# print(f"Train: {X_train.shape}, Test: {X_test.shape}")
# print(f"Rain prevalence — Train: {y_train.mean():.2%}, Test: {y_test.mean():.2%}")
print("Split logic ready — uncomment above cells to execute.")

## 4 — Model Instantiation Check

In [ ]:
# Verify all 4 models instantiate without error
models = get_all_models()
for name, pipeline in models.items():
    clf_name = type(pipeline.steps[-1][1]).__name__
    print(f"\u2705 {name}: {clf_name}")

## 5 — Cross-Validation Setup

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["roc_auc", "f1", "accuracy", "precision", "recall"]

# cv_results = {}
# for name, pipeline in models.items():
#     print(f"\nRunning CV for {name}...")
#     scores = cross_validate(
#         pipeline, X_train, y_train,
#         cv=cv_strategy, scoring=scoring,
#         n_jobs=-1, return_train_score=True
#     )
#     cv_results[name] = scores
#     print(f"  ROC-AUC: {scores['test_roc_auc'].mean():.4f} \u00b1 {scores['test_roc_auc'].std():.4f}")
#     print(f"  F1:      {scores['test_f1'].mean():.4f} \u00b1 {scores['test_f1'].std():.4f}")
#     print(f"  Accuracy:{scores['test_accuracy'].mean():.4f} \u00b1 {scores['test_accuracy'].std():.4f}")
print("CV strategy defined — uncomment above to run.")

## 6 — Fit & Save Models

In [ ]:
# for name, pipeline in models.items():
#     print(f"Fitting {name}...")
#     pipeline.fit(X_train, y_train)
#     save_path = os.path.join(MODELS_DIR, f"{name}.joblib")
#     joblib.dump(pipeline, save_path)
#     print(f"  Saved: {save_path}")
print("Fit + save logic ready — uncomment to execute.")

## 7 — ROC Curve Visualisation

In [ ]:
TRAINED = False  # Set to True after running training cells above

if TRAINED:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    for ax, (name, pipeline) in zip(axes.flat, models.items()):
        RocCurveDisplay.from_estimator(pipeline, X_test, y_test, ax=ax, name=name)
        ax.set_title(f"ROC Curve — {name}")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    plt.tight_layout()
    plt.savefig("artifacts/fig_roc_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("\u23ed Skipping plots — set TRAINED=True after fitting models.")

## 8 — Results Summary Table

In [ ]:
if TRAINED:
    summary_rows = []
    for name, pipeline in models.items():
        metrics = evaluate_model(pipeline, X_test, y_test)
        summary_rows.append({"Model": name, **metrics})

    summary_df = pd.DataFrame(summary_rows).set_index("Model")
    summary_df = summary_df.drop(columns=["confusion_matrix"])
    print(summary_df.sort_values("roc_auc", ascending=False).to_markdown())
else:
    print("\u23ed Skipping summary — set TRAINED=True after fitting models.")